<a href="https://colab.research.google.com/github/vidhya2026/Duffl_Career_Page/blob/main/Resume_SkillMatch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install PyMuPDF python-docx scikit-learn pandas openpyxl -q

In [33]:
!pip install docx2txt PyPDF2 scikit-learn

In [34]:
!pip install PyMuPDF python-docx sentence-transformers scikit-learn pandas openpyxl ipywidgets -q

In [35]:
import fitz
import docx
import io, re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output

print(" All libraries imported.")

 All libraries imported.


In [36]:
# Load the pretrained BERT model
# 'all-MiniLM-L6-v2' is a lightweight sentence transformer.


print(" Loading sentence-transformer model (first run downloads ~90MB)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(" Model loaded! Ready to embed text into 384-dimensional vectors.")

 Loading sentence-transformer model (first run downloads ~90MB)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Model loaded! Ready to embed text into 384-dimensional vectors.


In [37]:
#Text extraction helpers


def extract_pdf(file_bytes):
    text = ""
    try:
        doc = fitz.open(stream=file_bytes, filetype="pdf")
        for page in doc:
            text += page.get_text()
    except Exception as e:
        text = f"[PDF error: {e}]"
    return text.strip()

def extract_docx(file_bytes):
    text = ""
    try:
        doc = docx.Document(io.BytesIO(file_bytes))
        for para in doc.paragraphs:
            text += para.text + "\n"
    except Exception as e:
        text = f"[DOCX error: {e}]"
    return text.strip()

def extract_text(filename, file_bytes):
    ext = filename.lower().split(".")[-1]
    if ext == "pdf":
        return extract_pdf(file_bytes)
    elif ext in ("docx", "doc"):
        return extract_docx(file_bytes)
    elif ext == "txt":
        return file_bytes.decode("utf-8", errors="ignore")
    return ""

def clean_text(text):
    # Light cleaning — keep most content so BERT gets full context
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,\-]', ' ', text)
    return text.strip()

In [38]:

#Core matching logic (BERT + cosine similarity)


def get_semantic_score(jd_text, resume_text):
    """
    Converts both texts to 384-dim BERT embeddings.
    Then computes cosine similarity between the two vectors.
    Returns a score from 0 to 100.
    """
    embeddings = model.encode([jd_text, resume_text], convert_to_numpy=True)
    score = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return round(float(score) * 100, 2)


def get_missing_keywords(jd_text, resume_text, top_n=8):
    """
    Uses TF-IDF (still useful here!) to find important words
    in the JD that are completely absent from the resume.
    BERT tells us the overall score; this tells us WHAT is missing.
    """
    jd_clean   = clean_text(jd_text).lower()
    res_clean  = clean_text(resume_text).lower()
    try:
        vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=60)
        vec.fit([jd_clean])
        jd_keywords = vec.get_feature_names_out()
    except:
        return []
    resume_words = set(res_clean.split())
    missing = [kw for kw in jd_keywords if not set(kw.split()).intersection(resume_words)]
    return missing[:top_n]


def match_all(jd_text, resumes_dict, threshold):
    """
    Runs BERT semantic matching for all uploaded resumes against the JD.
    Returns a sorted DataFrame with score, status, and issues.
    """
    jd_clean = clean_text(jd_text)

    # Pre-embed the JD once (efficient — don't re-embed for every resume)
    jd_embedding = model.encode([jd_clean], convert_to_numpy=True)

    results = []
    total = len(resumes_dict)

    for i, (fname, raw_text) in enumerate(resumes_dict.items(), 1):
        print(f"  [{i}/{total}] Processing: {fname}")
        resume_clean = clean_text(raw_text)

        if len(resume_clean) < 30:
            results.append({
                "File": fname, "Score (%)": 0.0,
                "Status": "Not Matched",
                "Issues": "Could not extract text from file"
            })
            continue

        # BERT embedding for this resume
        resume_embedding = model.encode([resume_clean], convert_to_numpy=True)

        # Cosine similarity between JD vector and resume vector
        score = cosine_similarity(jd_embedding, resume_embedding)[0][0]
        score_pct = round(float(score) * 100, 2)

        if score_pct >= threshold:
            status = "Matched"
            issues = "—"
        else:
            status = "Not Matched"
            missing = get_missing_keywords(jd_text, raw_text)
            issues = f"Semantic similarity too low ({score_pct}%)"
            if missing:
                issues += f" | Possibly missing: {', '.join(missing)}"

        results.append({
            "File": fname,
            "Score (%)": score_pct,
            "Status": status,
            "Issues": issues
        })

    df = pd.DataFrame(results)
    df = df.sort_values("Score (%)", ascending=False).reset_index(drop=True)
    return df

In [39]:
#  HR enters Job Description
print("=" * 60)
print("  RESUME MATCHER — HR INPUT PANEL")
print("=" * 60)
print()
print("STEP 1: Paste the full Job Description below.")
print("        Then click 'Save JD & Continue'.\n")

jd_box = widgets.Textarea(
    placeholder=(
        "Paste the full job description here...\n\n"
        "Example:\nWe are looking for a Python Developer with experience in\n"
        "machine learning, deep learning, NLP, TensorFlow, SQL,\n"
        "REST APIs, AWS, and good communication skills."
    ),
    layout=widgets.Layout(width='100%', height='240px')
)

threshold_slider = widgets.IntSlider(
    value=50,
    min=10, max=90, step=5,
    description='Match threshold %:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='65%')
)

threshold_note = widgets.HTML(
    value="<i style='font-size:12px;color:gray;'>"
          "Tip: BERT scores are higher than TF-IDF. 50% is a good starting point. "
          "Raise it to 65%+ for stricter matching.</i>"
)

save_btn = widgets.Button(
    description='Save JD & Continue',
    button_style='primary',
    layout=widgets.Layout(width='210px')
)

status_out = widgets.Output()
jd_saved = {'text': None, 'threshold': 50}

def on_save(b):
    with status_out:
        clear_output()
        txt = jd_box.value.strip()
        if len(txt) < 30:
            print("  JD is too short. Please paste the full job description.")
            return
        jd_saved['text'] = txt
        jd_saved['threshold'] = threshold_slider.value
        print(f" JD saved — {len(txt)} characters.")
        print(f" Match threshold: {threshold_slider.value}%")

save_btn.on_click(on_save)

display(widgets.VBox([
    widgets.Label("Job Description:"),
    jd_box,
    threshold_slider,
    threshold_note,
    save_btn,
    status_out
]))




  RESUME MATCHER — HR INPUT PANEL

STEP 1: Paste the full Job Description below.
        Then click 'Save JD & Continue'.



In [41]:
# HR uploads resumes


print("  Upload candidate resumes.")
print("        Supported formats: PDF, DOCX, TXT")
print("        You can select multiple files at once.\n")

uploaded = files.upload()

resumes_dict = {}
for fname, fbytes in uploaded.items():
    text = extract_text(fname, fbytes)
    resumes_dict[fname] = text
    ok = "✓ text extracted" if len(text) > 30 else "⚠ empty / unreadable"
    print(f"  {fname:50s}  {ok}  ({len(text)} chars)")

print(f"\n Total resumes ready: {len(resumes_dict)}")
print("  Now run CELL 8 to start matching.")

  Upload candidate resumes.
        Supported formats: PDF, DOCX, TXT
        You can select multiple files at once.



Saving Naukri_DeepikaC[2y_6m].pdf to Naukri_DeepikaC[2y_6m].pdf
  Naukri_DeepikaC[2y_6m].pdf                          ✓ text extracted  (3312 chars)

 Total resumes ready: 1
  Now run CELL 8 to start matching.


In [42]:
#  Run BERT matching & show results


if not jd_saved['text']:
    print("  No JD found. Run CELL 6 first and click 'Save JD & Continue'.")

elif not resumes_dict:
    print(" No resumes found. Run CELL 7 first.")

else:
    threshold = jd_saved['threshold']
    print(f" Running BERT semantic matching (threshold = {threshold}%)...")
    print("   Each resume is converted to a 384-dim vector and compared to the JD.\n")

    results_df = match_all(jd_saved['text'], resumes_dict, threshold)

    # ── Results table
    print("\n" + "=" * 60)
    print("  RESULTS")
    print("=" * 60)
    pd.set_option('display.max_colwidth', 90)
    pd.set_option('display.width', 150)
    display(results_df)

    matched     = (results_df["Status"] == " Matched").sum()
    not_matched = len(results_df) - matched
    print(f"\n Summary: {matched} matched, {not_matched} not matched out of {len(results_df)} resumes")

    # ── Detailed breakdown ─────────────────────────────────
    print("\n" + "─" * 60)
    print("DETAILED BREAKDOWN")
    print("─" * 60)
    for _, row in results_df.iterrows():
        print(f"\n {row['File']}")
        print(f"   BERT Score : {row['Score (%)']}%")
        print(f"   Status     : {row['Status']}")
        if row['Status'] == " Not Matched":
            print(f"   Issues     : {row['Issues']}")

    # ── Download results
    csv_path  = "resume_results_bert.csv"
    xlsx_path = "resume_results_bert.xlsx"
    results_df.to_csv(csv_path, index=False)
    results_df.to_excel(xlsx_path, index=False)

    print("\n Downloading results...")
    files.download(csv_path)
    files.download(xlsx_path)
    print(" All done!")

 Running BERT semantic matching (threshold = 50%)...
   Each resume is converted to a 384-dim vector and compared to the JD.

  [1/1] Processing: Naukri_DeepikaC[2y_6m].pdf

  RESULTS


,File,Score (%),Status,Issues
0,Naukri_DeepikaC[2y_6m].pdf,55.05,Matched,—



 Summary: 0 matched, 1 not matched out of 1 resumes

────────────────────────────────────────────────────────────
DETAILED BREAKDOWN
────────────────────────────────────────────────────────────

 Naukri_DeepikaC[2y_6m].pdf
   BERT Score : 55.05%
   Status     : Matched



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 All done!


### **Flask API**

In [43]:
!pip install PyMuPDF python-docx sentence-transformers scikit-learn flask flask-cors pyngrok -q


In [44]:
import fitz
import docx
import io, re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading

print(" All imports done.")

app = Flask(__name__)
CORS(app)


 All imports done.


In [45]:
print("Loading BERT model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("BERT model ready.")


Loading BERT model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model ready.


In [46]:
def extract_pdf(file_bytes):
    text = ""
    try:
        doc = fitz.open(stream=file_bytes, filetype="pdf")
        for page in doc:
            text += page.get_text()
    except Exception as e:
        text = ""
    return text.strip()

def extract_docx(file_bytes):
    text = ""
    try:
        doc = docx.Document(io.BytesIO(file_bytes))
        for para in doc.paragraphs:
            text += para.text + "\n"
    except:
        text = ""
    return text.strip()

def extract_text(filename, file_bytes):
    ext = filename.lower().split(".")[-1]
    if ext == "pdf":
        return extract_pdf(file_bytes)
    elif ext in ("docx", "doc"):
        return extract_docx(file_bytes)
    elif ext == "txt":
        return file_bytes.decode("utf-8", errors="ignore")
    return ""

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,\-]', ' ', text)
    return text.strip()

def get_missing_keywords(jd_text, resume_text, top_n=8):
    jd_clean  = clean_text(jd_text).lower()
    res_clean = clean_text(resume_text).lower()
    try:
        vec = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=60)
        vec.fit([jd_clean])
        jd_keywords = vec.get_feature_names_out()
    except:
        return []
    resume_words = set(res_clean.split())
    missing = [kw for kw in jd_keywords if not set(kw.split()).intersection(resume_words)]
    return missing[:top_n]

def match_resume(jd_text, resume_text, filename,threshold):
    jd_clean     = clean_text(jd_text)
    resume_clean = clean_text(resume_text)

    if len(resume_clean) < 30:
        return {
            "file": filename,
            "score": 0.0,
            "status": "Not Matched",
            "issues": "Could not extract text from file"
        }

    jd_emb     = model.encode([jd_clean], convert_to_numpy=True)
    res_emb    = model.encode([resume_clean], convert_to_numpy=True)
    score      = cosine_similarity(jd_emb, res_emb)[0][0]
    score_pct  = round(float(score) * 100, 2)

    if score_pct >= threshold:
        status = "Matched"
        issues = ""
    else:
        status  = "Not Matched"
        missing = get_missing_keywords(jd_text, resume_text)
        issues  = f"Low similarity score ({score_pct}%)"
        if missing:
            issues += f". Possibly missing: {', '.join(missing)}"

    return {
        "file": filename,
        "score": score_pct,
        "status": status,
        "issues": issues
    }


In [47]:
# Flask API config
@app.route('/match', methods=['POST'])
def match():
    try:
        jd_text = request.form.get('jd', '').strip()


        threshold = request.form.get('threshold', 50)
        threshold = float(threshold)

        if len(jd_text) < 10:
            return jsonify({"error": "Job description is too short or missing."}), 400

        resume_files = request.files.getlist('resumes')
        if not resume_files:
            return jsonify({"error": "No resume files uploaded."}), 400

        results = []
        for f in resume_files:
            file_bytes = f.read()
            raw_text   = extract_text(f.filename, file_bytes)


            result = match_resume(jd_text, raw_text, f.filename, threshold)
            results.append(result)

        results.sort(key=lambda x: x['score'], reverse=True)

        matched     = sum(1 for r in results if r['status'] == 'Matched')
        not_matched = len(results) - matched

        return jsonify({
            "results": results,
            "summary": {
                "total": len(results),
                "matched": matched,
                "not_matched": not_matched
            }
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "running", "model": "all-MiniLM-L6-v2"})

In [48]:
!ps aux | grep ngrok

root       30047  0.0  0.0   7372  3432 ?        S    09:20   0:00 /bin/bash -c ps aux | grep ngrok
root       30049  0.0  0.0   6480  2416 ?        S    09:20   0:00 grep ngrok


In [50]:
!pkill -f flask
!pkill -f ngrok

from pyngrok import ngrok
ngrok.kill()

In [52]:
# Start Flask + expose via ngrok



NGROK_AUTH_TOKEN = "3BWStb5vBGBiPhdLT16BH7fv5PL_893euXis3Be3KEpnrjvrc"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Run Flask in a background thread (so Colab doesn't block)
def run_flask():
    app.run(port=5000)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()

# Expose port 5000 publicly
public_url = ngrok.connect(5000)
print("=" * 55)
print("   Flask API is LIVE!")
print(f"  Public URL : {public_url}")
print()
print("  Copy this URL and paste it into your C# app.")
print("  Your C# should POST to:")
print(f"  {public_url}/match")
print("=" * 55)

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


   Flask API is LIVE!
  Public URL : NgrokTunnel: "https://temperance-unmonetary-medianly.ngrok-free.dev" -> "http://localhost:5000"

  Copy this URL and paste it into your C# app.
  Your C# should POST to:
  NgrokTunnel: "https://temperance-unmonetary-medianly.ngrok-free.dev" -> "http://localhost:5000"/match
